<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 Nano Action: Inverse Dynamics with SGLang

This notebook runs Cosmos3 Nano **action inverse-dynamics** inference through the SGLang OpenAI-compatible action generation API:

```text
POST /v1/actions/generations
```

Inverse dynamics is the reverse of forward dynamics: given a video, it predicts the ego-motion (action) trajectory that produced it. This notebook builds the same custom input spec as [`run_id_with_cosmos_framework.ipynb`](./run_id_with_cosmos_framework.ipynb), keeps the same input-video preview and predicted-trajectory visualization, and only changes the environment setup plus the inference call.


### Set up HF Cache:

SGLang container is run under root by default. User can create a writable cache directory on host, or start the container with host user id.

```bash
export SGLANG_HF_CACHE="${SGLANG_HF_CACHE:-$HOME/.cache/sglang-huggingface}"
mkdir -p "$SGLANG_HF_CACHE"
chmod 777 "$SGLANG_HF_CACHE"
```

### Start SGLang Server

Start the server in a terminal from the `cosmos` repo root.

```bash
docker rm -f cosmos3-sglang-notebook 2>/dev/null || true

docker run -d --init --name cosmos3-sglang-notebook \
  --runtime nvidia --gpus '"device=0"' \
  -e CUDA_DEVICE_ORDER=PCI_BUS_ID \
  -v $SGLANG_HF_CACHE:/root/.cache/huggingface \
  -v "$PWD:/workspace" \
  -p 30000:30000 --ipc=host \
  lmsysorg/sglang:dev \
  sglang serve \
  --model-path nvidia/Cosmos3-Nano \
  --host 0.0.0.0

# Wait until this returns model metadata before running the inference cell.
curl http://localhost:30000/v1/models
```


## Configure Notebook Variables

Before running the variables cell, set `COSMOS3_REPO` to the root of a local checkout of the **cosmos-framework** repository:

```bash
export COSMOS3_REPO=/path/to/cosmos-framework
```

This must be the framework checkout that contains the `cosmos_framework/` Python package—not this `cosmos` cookbook repository, a model directory, or the `packages/cosmos3` model sources. The notebook adds this path to Python's import path to load `cosmos_framework`. Set it in the shell that launches Jupyter before running the variables cell.


In [ ]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", COSMOS_ROOT)).resolve()
COSMOS3_FRAMEWORK_PATH = Path(
    os.environ.get("COSMOS3_FRAMEWORK_PATH", COSMOS_ROOT.parent / "cosmos-framework")
).resolve()
COSMOS3_OUTPUT_ROOT = Path(
    os.environ.get("COSMOS3_SGLANG_OUTPUT_ROOT", COSMOS_ROOT / "outputs" / "cosmos3_action_sglang")
).resolve()
COSMOS3_INPUT_DIR = COSMOS3_OUTPUT_ROOT / "inputs"
SGLANG_BASE_URL = os.environ.get("COSMOS3_SGLANG_BASE_URL", "http://localhost:30000").rstrip("/")

COSMOS3_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
COSMOS3_INPUT_DIR.mkdir(parents=True, exist_ok=True)

print("COSMOS_ROOT:", COSMOS_ROOT)
print("COSMOS3_REPO:", COSMOS3_REPO)
print("COSMOS3_FRAMEWORK_PATH:", COSMOS3_FRAMEWORK_PATH)
print("COSMOS3_INPUT_DIR:", COSMOS3_INPUT_DIR)
print("COSMOS3_OUTPUT_ROOT:", COSMOS3_OUTPUT_ROOT)
print("COSMOS3_SGLANG_BASE_URL:", SGLANG_BASE_URL)


## Create the Inverse-Dynamics Input Spec

Inverse-dynamics inference is driven by a JSONL spec, one line per run. Unlike forward dynamics, each line provides only an input video (`vision_path`) and **no** `action_path` — the action is what the model predicts.

This cell builds that spec from local AV videos, writing it under:

```text
outputs/cosmos3_action_sglang/inputs/action_inverse_dynamics_av_custom.jsonl
```

It mirrors the native PyTorch notebook's spec format. The `vision_path` is written as an absolute path, because the SGLang request cell reads the spec records directly.


In [ ]:
# `os` and the COSMOS3_* paths come from the configuration cell.
import json

# Local inputs, relative to the cosmos repo root.
input_videos = {
    "av_inverse_0": "cookbooks/cosmos3/generator/action/assets/videos/av_0.mp4",
    "av_inverse_1": "cookbooks/cosmos3/generator/action/assets/videos/av_1.mp4",
}

def resolve_input(rel_path: str) -> str:
    path = (COSMOS_ROOT / rel_path).resolve()
    assert path.exists(), f"missing input: {path}"
    return str(path)

records = [
    {
        "action_chunk_size": 60,
        "domain_name": "av",
        "fps": 10,
        "image_size": 480,
        "view_point": "ego_view",
        "model_mode": "inverse_dynamics",
        "name": name,
        "prompt": "You are an autonomous vehicle planning system.",
        "seed": 0,
        "vision_path": resolve_input(video_rel),
    }
    for name, video_rel in input_videos.items()
]

COSMOS3_INPUT_DIR.mkdir(parents=True, exist_ok=True)
id_input_path = COSMOS3_INPUT_DIR / "action_inverse_dynamics_av_custom.jsonl"
id_input_path.write_text("".join(json.dumps(r) + "\n" for r in records))
id_output_dir = COSMOS3_OUTPUT_ROOT / "action_inverse_dynamics_av_custom"

# The bash inference cell can only see the environment, so export the paths it needs.
os.environ["COSMOS3_ID_INPUT"] = str(id_input_path)
os.environ["COSMOS3_ID_OUTPUT"] = str(id_output_dir)

print("wrote spec:", id_input_path)
print("runs:", list(input_videos))
print(id_input_path.read_text())

## Preview the Input Video(s)

Preview each input video before running inference. `Video(..., embed=True)` base64-inlines the file, and these AV clips are several MB each, so we first transcode a small preview (~150 KB) with the ffmpeg binary bundled in `imageio-ffmpeg` (installed by `uv sync`), then embed it. The original input videos are left untouched.

In [ ]:
import subprocess
import imageio_ffmpeg
from IPython.display import Video, display

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

def make_preview(src: Path, dst: Path, crf: int = 28) -> Path:
    """Re-encode `src` to a compact, browser-friendly mp4 (cached)."""
    if not dst.exists():
        subprocess.run(
            [FFMPEG, "-y", "-loglevel", "error", "-i", str(src),
             "-c:v", "libx264", "-crf", str(crf),
             "-preset", "veryfast", "-an", "-pix_fmt", "yuv420p", str(dst)],
            check=True,
        )
    return dst

# `records` comes from the prepare cell; preview each input video.
for record in records:
    name = record["name"]
    src = Path(record["vision_path"])
    preview = make_preview(src, COSMOS3_INPUT_DIR / f"{name}_input_preview.mp4")
    print(f"{name}  ({src.stat().st_size // 1024} KB -> {preview.stat().st_size // 1024} KB preview)")
    display(Video(str(preview), embed=True))

## Run Inverse-Dynamics Inference

Runs `Cosmos3-Nano` on every line of the spec through SGLang. Inverse dynamics predicts an action, and this cell writes a PyTorch-compatible result file for each run:

```text
<output>/<name>/sample_outputs.json
```

The predicted action trajectory is stored under `outputs[0].content["action"]`, matching the native notebook's visualization cell. SGLang also returns `response.json`, `final.json`, and `action.json` for API debugging.


In [ ]:
import json
import time
from pathlib import Path

try:
    import requests
except ImportError as exc:
    raise RuntimeError("Install requests in this notebook kernel: pip install requests") from exc


def check_sglang_server() -> None:
    response = requests.get(f"{SGLANG_BASE_URL}/v1/models", timeout=10)
    response.raise_for_status()
    print(response.json())


def log_sglang_error(response: requests.Response, run_dir: Path, label: str, context: dict | None = None) -> None:
    error_path = run_dir / f"{label}_error_response.txt"
    error_path.write_text(response.text)
    print(f"SGLang {label} request failed:", response.status_code)
    print(response.text)
    print("saved error response:", error_path)
    if context is not None:
        print("context:", json.dumps(context, indent=2))


def submit_inverse_dynamics(record: dict) -> dict:
    run_dir = id_output_dir / record["name"]
    run_dir.mkdir(parents=True, exist_ok=True)

    video_path = Path(record["vision_path"])
    extra_params = {
        "action_mode": "inverse_dynamics",
        "domain_name": record["domain_name"],
        "action_view_point": record["view_point"],
        "raw_action_dim": 9,
        "guardrails": False,
    }
    form = {
        "prompt": record["prompt"],
        "num_frames": record["action_chunk_size"] + 1,
        "fps": record["fps"],
        "num_inference_steps": 30,
        "guidance_scale": 1.0,
        "flow_shift": 10.0,
        "seed": record["seed"],
        "extra_params": json.dumps(extra_params),
    }

    with video_path.open("rb") as video_file:
        response = requests.post(
            f"{SGLANG_BASE_URL}/v1/actions/generations",
            data={key: str(value) for key, value in form.items()},
            files={"input_reference": (video_path.name, video_file, "video/mp4")},
            timeout=120,
        )
    if not response.ok:
        log_sglang_error(
            response,
            run_dir,
            "submit",
            {"form": form, "extra_params": extra_params, "video_path": str(video_path)},
        )
        response.raise_for_status()
    response.raise_for_status()
    action = response.json()['data'][0]['action']
    (run_dir / "action.json").write_text(json.dumps(action, indent=2))

    sample_outputs = {"outputs": [{"content": {"action": action.get("values")}}]}
    (run_dir / "sample_outputs.json").write_text(json.dumps(sample_outputs, indent=2))

    print("saved", run_dir / "sample_outputs.json")
    print("action shape:", action.get("shape"), "dtype:", action.get("dtype"))
    return {"record": record, "run_dir": run_dir, "action": action}


check_sglang_server()
results = []
for record in records:
    print(f"\nSubmitting {record['name']}")
    results.append(submit_inverse_dynamics(record))


## Visualize the Predicted Action

Plot the action the model predicted from each input video, as a 3D camera path (with frustums) and a top-down bird's-eye view. The action is read from each run's `sample_outputs.json` and interpreted with the AV pose convention.

In [ ]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection

# The notebook kernel may differ from the framework venv, so put the repo on the
# path before importing `cosmos_framework`.
if str(COSMOS3_FRAMEWORK_PATH) not in sys.path:
    sys.path.insert(0, str(COSMOS3_FRAMEWORK_PATH))
from cosmos_framework.data.generator.action.utils.pose_utils import pose_rel_to_abs

# frustum: apex + image-rectangle corners (camera +Z forward), and their edges
_FRUSTUM = np.array([[0, 0, 0], [-1, -1, 1], [1, -1, 1], [1, 1, 1], [-1, 1, 1]], float)
_EDGES = [(0, 1), (0, 2), (0, 3), (0, 4), (1, 2), (2, 3), (3, 4), (4, 1)]

def visualize_pose(poses_abs, *, n_frustums=20, scale_frac=0.03, aspect=16 / 9,
                   fov_deg=60.0, vertical_exaggeration=1.0, cmap="turbo",
                   title=None, save_path=None, show=True):
    """3D camera trajectory (with frustums) + a top-down bird's-eye view.

    AV convention: world Y is up, world +Z is the heading. `vertical_exaggeration`
    stretches only the up-axis box (uniform world scaling, so frustums never skew);
    1.0 = geometrically faithful. The 3D plot reorders world (X, Y, Z) -> (X, Z, Y)
    so Y points up on screen.
    """
    poses_abs = np.asarray(poses_abs)
    pos = poses_abs[:, :3, 3]                     # camera centers [T, 3]
    fwd = poses_abs[:, :3, 2]                     # heading (+Z) [T, 3]
    T = len(pos)
    colors = plt.get_cmap(cmap)(np.arange(T) / max(T - 1, 1))
    scale = max(np.ptp(pos, axis=0).max() * scale_frac, 1e-3)
    step = max(1, T // max(n_frustums, 1))
    xzy = [0, 2, 1]                               # world (X,Y,Z) -> plot (X, Z, Y-up)

    fig = plt.figure(figsize=(14, 6))

    # (1) 3D perspective with frustums
    ax = fig.add_subplot(1, 2, 1, projection="3d")
    path = pos[:, xzy]
    ax.plot(*path.T, color="0.6", lw=1.0, alpha=0.7)
    lines, lcolors, allpts = [], [], [path]
    for i in range(0, T, step):
        cw = ((_FRUSTUM * [aspect, 1, 1] * scale * np.tan(np.radians(fov_deg) / 2))
              @ poses_abs[i, :3, :3].T + poses_abs[i, :3, 3])[:, xzy]  # frustum in plot coords
        allpts.append(cw)
        lines += [[cw[a], cw[b]] for a, b in _EDGES]
        lcolors += [colors[i]] * len(_EDGES)
    ax.add_collection3d(Line3DCollection(lines, colors=lcolors, linewidths=1.2))
    ax.scatter(*path[0], color="lime", s=80, edgecolor="k", label="first frame", zorder=5)
    ax.scatter(*path[-1], color="red", s=80, edgecolor="k", label="last frame", zorder=5)
    rng = np.clip(np.ptp(np.concatenate(allpts), axis=0), 1e-9, None)
    ax.set_box_aspect((rng[0], rng[1], rng[2] * vertical_exaggeration))
    ax.set_xlabel("X (m)", labelpad=12); ax.set_ylabel("Z forward (m)", labelpad=12)
    ax.set_zlabel("Y up (m)", labelpad=10); ax.set_zticks([])
    ax.set_title(title or f"Camera trajectory + frustums ({T} frames)")
    ax.legend(loc="upper left"); ax.view_init(elev=22, azim=-70)

    # (2) top-down bird's-eye view (X-Z ground plane)
    ax2 = fig.add_subplot(1, 2, 2)
    seg = np.stack([pos[:-1, [0, 2]], pos[1:, [0, 2]]], axis=1)
    lc = LineCollection(seg, cmap=cmap, norm=plt.Normalize(0, T - 1), linewidth=2.5)
    lc.set_array(np.arange(T - 1)); ax2.add_collection(lc)
    ax2.quiver(pos[::step, 0], pos[::step, 2], fwd[::step, 0], fwd[::step, 2],
               color=colors[::step], angles="xy", width=0.005, scale=22, zorder=3)
    ax2.scatter(*pos[0, [0, 2]], color="lime", s=80, edgecolor="k", label="first frame", zorder=5)
    ax2.scatter(*pos[-1, [0, 2]], color="red", s=80, edgecolor="k", label="last frame", zorder=5)
    ax2.set_xlabel("X (m)"); ax2.set_ylabel("Z forward (m)")
    ax2.set_title("Top-down (bird's-eye view)")
    ax2.set_aspect("equal", adjustable="datalim"); ax2.autoscale_view(); ax2.legend()
    fig.colorbar(lc, ax=ax2, label="frame index")

    plt.tight_layout(w_pad=6)
    if save_path:
        fig.savefig(save_path, dpi=120, bbox_inches="tight"); print("saved", save_path)
    if show:
        plt.show()

# `records` and `id_output_dir` come from the prepare cell; read each run's
# predicted action from its sample_outputs.json.
for record in records:
    name = record["name"]
    outputs = json.loads((id_output_dir / name / "sample_outputs.json").read_text())
    poses_rel = np.array(outputs["outputs"][0]["content"]["action"])  # [T-1, 9] = [translation(3), rot6d(6)]

    # AV action convention (see cosmos_framework/data/generator/action/utils/pose_utils.py):
    # rot6d rotation, backward_framewise, translation_scale = 1.35.
    poses_abs = pose_rel_to_abs(
        poses_rel,
        rotation_format="rot6d",
        pose_convention="backward_framewise",
        translation_scale=1.35,
    )  # [T, 4, 4] camera-to-world
    print(name, poses_rel.shape, poses_abs.shape)
    visualize_pose(poses_abs, title=f"{name}: predicted camera trajectory + frustums ({len(poses_abs)} frames)", show=True)

## Cosmos3-Nano-HumanAction: Inverse Dynamics for human hands and body

[Cosmos3-Nano-HumanAction](https://huggingface.co/nvidia/Cosmos3-Nano-HumanAction) is Cosmos3-Nano post-trained on human
egocentric data. It serves three action domains from one checkpoint:

| case | data | `domain_name` (id) | action | layout |
| --- | --- | --- | --- | --- |
| `humanaction_hand` | Action100M hands (`assets/webhumanaction_hand_lerobot_example`) | `webhumanaction_hand` (31) | 48D | right wrist (9) + right fingertips (15) + left wrist (9) + left fingertips (15) |
| `humanaction_body` | Action100M body (`assets/webhumanaction_body_lerobot_example`) | `webhumanaction_body` (24) | 57D | head (9) + the same 48D hand block |
| `humanaction_mecka` | Mecka hands (`assets/human_hand_pose_lerobot_example`) | `hand_pose` (3) | 57D | camera (9) + the same 48D hand block |

Recipe (identical to post-training): 15 fps, 72-step chunks (73 frames), 480p tier, `backward_chunk_anchored_16f` rot6d
deltas, `piecewise_asinh_rot` normalization with the per-domain stats shipped in cosmos-framework under
`normalizer_stats/`, prompt = the episode's task/subtask caption, 30 steps, guidance 1.0, flow shift 5.0. The readers
`WebHumanActionHandLeRobotDataset`, `WebHumanActionBodyLeRobotDataset` and `HumanHandPoseLeRobotDataset` (with the
HumanAction arguments) build exactly this representation from LeRobot v3 data; `dataset.denormalize()` inverts it.

**SGLang notes.** One server serves one model: restart the container from the server cell above with `--model-path $COSMOS3_HUMANACTION_MODEL` (default `nvidia/Cosmos3-Nano-HumanAction`) before running this section. Forward dynamics goes through `POST /v1/videos`; inverse dynamics and policy go through `POST /v1/actions/generations`, which returns actions only, so the policy rollout video is rendered by a second forward-dynamics request (the same two-step pattern as the DROID policy notebook). The reader cell needs a cosmos-framework checkout with the HumanAction readers on `COSMOS3_FRAMEWORK_PATH`.


### Cases

In [ ]:
# ---- Cosmos3-Nano-HumanAction: shared case definitions (Action100M hands / Action100M body / Mecka hands) ----
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image as PILImage

# The public LeRobot readers live in cosmos-framework; put the checkout on sys.path like the other sections do.
_ha_framework = globals().get("COSMOS3_FRAMEWORK_PATH") or globals().get("COSMOS3_REPO")
if _ha_framework is not None and str(_ha_framework) not in sys.path:
    sys.path.insert(0, str(_ha_framework))
from cosmos_framework.data.generator.action.datasets import (
    HumanHandPoseLeRobotDataset,
    WebHumanActionBodyLeRobotDataset,
    WebHumanActionHandLeRobotDataset,
)
from cosmos_framework.data.generator.action.datasets.human_hand_pose_lerobot_dataset import HUMANACTION_NORMALIZER_PATH
from cosmos_framework.data.generator.action.utils.pose_utils import pose_rel_to_abs

HUMANACTION_MODEL_ID = os.environ.get("COSMOS3_HUMANACTION_MODEL", "nvidia/Cosmos3-Nano-HumanAction")
HUMANACTION_ASSETS = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "action" / "assets"
HUMANACTION_OUTPUT_ROOT = (
    Path(os.environ.get("COSMOS3_ACTION_OUTPUT_ROOT", str(globals().get("COSMOS3_OUTPUT_ROOT", COSMOS_ROOT / "outputs"))))
    / "humanaction"
)
HUMANACTION_SAMPLING = {"num_steps": 30, "guidance": 1.0, "shift": 5.0, "seed": 0}
HUMANACTION_FPS = 15
HUMANACTION_CHUNK = 72
HUMANACTION_RESOLUTION_TIER = 480
HUMANACTION_POSE_CONVENTION = "backward_chunk_anchored_16f"

# wrist_slice: the right-wrist 9D pose block used for the trajectory plots.
HUMANACTION_CASES = {
    "humanaction_hand": {
        "label": "Action100M hands",
        "domain_name": "webhumanaction_hand",
        "domain_id": 31,
        "raw_action_dim": 48,
        "root": "webhumanaction_hand_lerobot_example",
        "wrist_slice": slice(0, 9),
    },
    "humanaction_body": {
        "label": "Action100M body (head + hands)",
        "domain_name": "webhumanaction_body",
        "domain_id": 24,
        "raw_action_dim": 57,
        "root": "webhumanaction_body_lerobot_example",
        "wrist_slice": slice(9, 18),
    },
    "humanaction_mecka": {
        "label": "Mecka hands (camera + hands)",
        "domain_name": "hand_pose",
        "domain_id": 3,
        "raw_action_dim": 57,
        "root": "human_hand_pose_lerobot_example",
        "wrist_slice": slice(9, 18),
    },
}


def humanaction_dataset(case: str):
    root = str(HUMANACTION_ASSETS / HUMANACTION_CASES[case]["root"])
    if case == "humanaction_hand":
        return WebHumanActionHandLeRobotDataset(root)
    if case == "humanaction_body":
        return WebHumanActionBodyLeRobotDataset(root)
    # Same Mecka-format data as the base Cosmos3-Nano hand-pose example, read with the HumanAction recipe.
    return HumanHandPoseLeRobotDataset(
        root,
        chunk_length=HUMANACTION_CHUNK,
        pose_convention=HUMANACTION_POSE_CONVENTION,
        action_normalization="piecewise_asinh_rot",
        stats_path=HUMANACTION_NORMALIZER_PATH,
    )


def load_humanaction_case(case: str, index: int = 0) -> dict:
    """One 73-frame window: uint8 frames, normalized ground-truth actions, caption, and an output dir."""
    dataset = humanaction_dataset(case)
    sample = dataset[index]
    frames = sample["video"].permute(1, 2, 3, 0).cpu().numpy()  # [T,H,W,3] uint8
    output_dir = HUMANACTION_OUTPUT_ROOT / case
    output_dir.mkdir(parents=True, exist_ok=True)
    return {
        "case": case,
        **HUMANACTION_CASES[case],
        "dataset": dataset,
        "sample": sample,
        "actions": sample["action"].float(),
        "frames": frames,
        "first_frame": PILImage.fromarray(frames[0]),
        "caption": sample["ai_caption"],
        "fps": int(sample["conditioning_fps"]),
        "chunk": int(sample["action"].shape[0]),
        "view_point": sample["viewpoint"],
        "output_dir": output_dir,
    }


def write_humanaction_clip(frames: np.ndarray, path: Path, fps: int) -> Path:
    """Encode uint8 [T,H,W,3] frames as an h264 mp4 (PyAV)."""
    import av

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with av.open(str(path), mode="w") as container:
        stream = container.add_stream("libx264", rate=fps)
        stream.width, stream.height, stream.pix_fmt = int(frames.shape[2]), int(frames.shape[1]), "yuv420p"
        stream.options = {"crf": "18", "preset": "veryfast"}
        for frame in frames:
            for packet in stream.encode(av.VideoFrame.from_ndarray(np.ascontiguousarray(frame), format="rgb24")):
                container.mux(packet)
        for packet in stream.encode():
            container.mux(packet)
    return path


def humanaction_wrist_poses(case_data: dict, actions_norm) -> np.ndarray:
    """Denormalize a (predicted or ground-truth) chunk and integrate the right-wrist block into absolute poses."""
    raw = case_data["dataset"].denormalize(torch.as_tensor(np.asarray(actions_norm), dtype=torch.float32))
    poses_rel = raw[:, case_data["wrist_slice"]].cpu().numpy().astype(np.float64)
    return pose_rel_to_abs(poses_rel, rotation_format="rot6d", pose_convention=HUMANACTION_POSE_CONVENTION)


def humanaction_action_error(case_data: dict, pred_norm) -> float:
    """Mean squared error in the model's normalized action space against the reader's ground truth."""
    pred = torch.as_tensor(np.asarray(pred_norm), dtype=torch.float32)
    return float(((pred - case_data["actions"]) ** 2).mean())


def humanaction_report_actions(case_data: dict, pred_norm, title: str) -> None:
    pred = np.asarray(pred_norm, dtype=np.float32)
    print(f"{title}: predicted {pred.shape[0]} steps x {pred.shape[1]}D, normalized MSE vs ground truth = {humanaction_action_error(case_data, pred):.4f}")
    if "visualize_pose" in globals():
        visualize_pose(humanaction_wrist_poses(case_data, pred), title=f"{case_data['case']}: {title}, right wrist (predicted)")
        visualize_pose(humanaction_wrist_poses(case_data, case_data["actions"]), title=f"{case_data['case']}: right wrist (ground truth)")


for _case in HUMANACTION_CASES:
    _data = load_humanaction_case(_case)
    print(f"{_case:18s} {_data['label']:32s} domain {_data['domain_name']} ({_data['domain_id']}) "
          f"actions {tuple(_data['actions'].shape)} video {_data['frames'].shape} fps {_data['fps']} caption: {_data['caption'][:70]!r}")


### Run (`/v1/actions/generations`)

In [ ]:
import mimetypes

try:
    import requests
except ImportError as exc:
    raise RuntimeError("Install requests in this notebook kernel: pip install requests") from exc


def humanaction_sglang_video(data: dict, media_path: Path, action) -> Path:
    """Forward dynamics through POST /v1/videos (async job, then download the content)."""
    run_dir = data["output_dir"] / "sglang_forward_dynamics"
    run_dir.mkdir(parents=True, exist_ok=True)
    width, height = data["first_frame"].size
    extra_params = {
        "action_mode": "forward_dynamics",
        "domain_name": data["domain_name"],
        "domain_id": data["domain_id"],
        "raw_action_dim": data["raw_action_dim"],
        "action_chunk_size": data["chunk"],
        "action_view_point": data["view_point"],
        "action": np.asarray(action, dtype=np.float32).tolist(),
        "guardrails": False,
    }
    form = {
        "prompt": data["caption"],
        "num_frames": data["chunk"] + 1,
        "fps": data["fps"],
        "size": f"{width}x{height}",
        "num_inference_steps": HUMANACTION_SAMPLING["num_steps"],
        "guidance_scale": HUMANACTION_SAMPLING["guidance"],
        "flow_shift": HUMANACTION_SAMPLING["shift"],
        "seed": HUMANACTION_SAMPLING["seed"],
        "extra_params": json.dumps(extra_params),
    }
    mime = mimetypes.guess_type(media_path.name)[0] or "application/octet-stream"
    with open(media_path, "rb") as fh:
        response = requests.post(
            f"{SGLANG_BASE_URL}/v1/videos",
            data={key: str(value) for key, value in form.items()},
            files={"input_reference": (media_path.name, fh, mime)},
            timeout=300,
        )
    if not response.ok:
        (run_dir / "error_response.txt").write_text(response.text)
        print("SGLang request failed:", response.status_code, response.text[:800])
        response.raise_for_status()
    job = response.json()
    while True:
        final = requests.get(f"{SGLANG_BASE_URL}/v1/videos/{job['id']}", timeout=60).json()
        if final.get("status") == "completed":
            break
        if final.get("status") in {"failed", "cancelled"}:
            raise RuntimeError(json.dumps(final, indent=2)[:2000])
        time.sleep(3)
    content = requests.get(f"{SGLANG_BASE_URL}/v1/videos/{job['id']}/content", timeout=300)
    content.raise_for_status()
    video_path = run_dir / f"{data['case']}_forward_dynamics.mp4"
    video_path.write_bytes(content.content)
    return video_path


def humanaction_sglang_actions(data: dict, action_mode: str, media_path: Path) -> np.ndarray:
    """Inverse dynamics / policy through POST /v1/actions/generations (synchronous, actions only)."""
    run_dir = data["output_dir"] / f"sglang_{action_mode}"
    run_dir.mkdir(parents=True, exist_ok=True)
    width, height = data["first_frame"].size
    extra_params = {
        "action_mode": action_mode,
        "domain_name": data["domain_name"],
        "domain_id": data["domain_id"],
        "raw_action_dim": data["raw_action_dim"],
        "action_chunk_size": data["chunk"],
        "action_view_point": data["view_point"],
        "guardrails": False,
    }
    form = {
        "prompt": data["caption"],
        "num_frames": data["chunk"] + 1,
        "fps": data["fps"],
        "height": height,
        "width": width,
        "num_inference_steps": HUMANACTION_SAMPLING["num_steps"],
        "guidance_scale": HUMANACTION_SAMPLING["guidance"],
        "flow_shift": HUMANACTION_SAMPLING["shift"],
        "seed": HUMANACTION_SAMPLING["seed"],
        "extra_params": json.dumps(extra_params),
    }
    mime = mimetypes.guess_type(media_path.name)[0] or "application/octet-stream"
    with open(media_path, "rb") as fh:
        response = requests.post(
            f"{SGLANG_BASE_URL}/v1/actions/generations",
            data={key: str(value) for key, value in form.items()},
            files={"input_reference": (media_path.name, fh, mime)},
            timeout=600,
        )
    if not response.ok:
        (run_dir / "error_response.txt").write_text(response.text)
        print("SGLang request failed:", response.status_code, response.text[:800])
        response.raise_for_status()
    payload = response.json()
    (run_dir / "response.json").write_text(json.dumps(payload, indent=2)[:200000])
    action = payload["data"][0]["action"]
    values = action.get("values") or action.get("data")
    pred = np.asarray(values, dtype=np.float32)
    if pred.ndim == 3:
        pred = pred[0]
    pred = pred[:, : data["raw_action_dim"]]
    (run_dir / "action.json").write_text(json.dumps(pred.tolist()) + "\n")
    return pred


humanaction_id_results = {}
for case in HUMANACTION_CASES:
    data = load_humanaction_case(case)
    data["input_clip"] = write_humanaction_clip(data["frames"], data["output_dir"] / f"{case}_input.mp4", data["fps"])
    data["pred"] = humanaction_sglang_actions(data, "inverse_dynamics", data["input_clip"])
    humanaction_id_results[case] = data


### View Results

In [ ]:
from IPython.display import Video, display

for case, data in humanaction_id_results.items():
    print(f"== {case} ({data['label']}): {data['caption']}")
    display(Video(str(data["input_clip"]), embed=True, width=480))
    humanaction_report_actions(data, data["pred"], "inverse dynamics")
